목표 : 예약 취소에 따른 객실 운영의 불확실성 줄이기

- `daily_rate`는 보존합니다. 이상치 처리 결과는 `daily_rate_processed`에 저장하고, `estimated_room_amount`는 `daily_rate`로 계산합니다.

- 미취소 예약은 기준 시점에 취소되지 않은 예약이며, 최종 투숙 완료 예약이 아닙니다. 관측 취소율을 최종 취소율이나 인과관계로 단정하지 않습니다.

`hotel_bookings.csv`는 교육용으로 제작한 가상의 호텔 예약 데이터이며, 결측치와 오류 값이 일부 포함되어 있습니다. 한 행은 하나의 예약을 의미합니다.

| 열 이름 | 설명 |
| --- | --- |
| `booking_id` | 예약 식별 번호 |
| `lead_time` | 예약일부터 예정 입실일까지의 일수 |
| `weekend_nights` | 주말 숙박일수 |
| `weekday_nights` | 평일 숙박일수 |
| `adults` | 성인 인원수 |
| `children` | 어린이 인원수 |
| `room_type` | 객실 유형: Standard, Deluxe, Suite |
| `daily_rate` | 예약에 적용된 1박당 객실 요금, 단위: 원 |
| `booking_channel` | 예약 경로: Direct, Online, Agency |
| `is_canceled` | 취소 여부: 0=미취소, 1=취소 |

**데이터 및 업무 조건**

- 예약 번호는 중복되지 않으며, `is_canceled`에는 결측치가 없습니다.
- `lead_time`, 숙박일수, 인원수에는 음수가 올 수 없습니다.
- 이 호텔은 무료 숙박을 취급하지 않으므로 `daily_rate`는 0보다 커야 합니다.
- `children`의 결측값은 ‘어린이 0명’을 의미하지 않습니다.
- 객실 유형에 따라 요금이 다르므로, 높은 요금이 반드시 오류는 아닙니다.
- 미취소 예약은 기준 시점에 취소되지 않은 예약이며, 최종 투숙 완료를 의미하지 않습니다.

In [271]:
import pandas as pd
import numpy as np

# 1

In [272]:
booking_raw = pd.read_csv("./hotel_bookings.csv")
booking_raw.head()

,booking_id,lead_time,weekend_nights,weekday_nights,adults,children,room_type,daily_rate,booking_channel,is_canceled
0,HB0204,15.0,2.0,5.0,2.0,1.0,Deluxe,180000.0,Direct,0
1,HB0267,7.0,0.0,4.0,2.0,0.0,Standard,130000.0,Direct,0
2,HB0153,28.0,2.0,4.0,1.0,0.0,Standard,70000.0,Online,0
3,HB0010,30.0,2.0,3.0,2.0,2.0,Standard,130000.0,Online,0
4,HB0234,171.0,0.0,4.0,2.0,1.0,Standard,120000.0,Agency,0


In [273]:
len(booking_raw)

300

In [274]:
len(booking_raw.columns)

10

In [275]:
booking_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   booking_id       300 non-null    object 
 1   lead_time        298 non-null    float64
 2   weekend_nights   298 non-null    float64
 3   weekday_nights   298 non-null    float64
 4   adults           298 non-null    float64
 5   children         298 non-null    float64
 6   room_type        296 non-null    object 
 7   daily_rate       296 non-null    float64
 8   booking_channel  296 non-null    object 
 9   is_canceled      300 non-null    int64  
dtypes: float64(6), int64(1), object(3)
memory usage: 23.6+ KB


In [276]:
booking_raw.describe()

,lead_time,weekend_nights,weekday_nights,adults,children,daily_rate,is_canceled
count,298.000000,298.000000,298.000000,298.000000,298.000000,296.000000,300.000000
mean,47.677852,0.942953,2.953020,1.768456,0.422819,166587.837838,0.233333
std,51.739801,0.812425,1.448722,0.605873,0.683438,137061.828387,0.423659
min,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-10000.000000,0.000000
25%,8.000000,0.000000,2.000000,1.000000,0.000000,100000.000000,0.000000
50%,23.000000,1.000000,3.000000,2.000000,0.000000,120000.000000,0.000000
75%,78.750000,2.000000,4.000000,2.000000,1.000000,180000.000000,0.000000
max,179.000000,2.000000,5.000000,3.000000,2.000000,1000000.000000,1.000000


In [277]:
booking_raw.isna().sum()

booking_id         0
lead_time          2
weekend_nights     2
weekday_nights     2
adults             2
children           2
room_type          4
daily_rate         4
booking_channel    4
is_canceled        0
dtype: int64

In [278]:
#열별 결측치개수와 결측률 표로 작성
booking_raw.isna().sum()

booking_id         0
lead_time          2
weekend_nights     2
weekday_nights     2
adults             2
children           2
room_type          4
daily_rate         4
booking_channel    4
is_canceled        0
dtype: int64

In [279]:

missing_rate = booking_raw.isna().mean() * 100
print(missing_rate)

booking_id         0.000000
lead_time          0.666667
weekend_nights     0.666667
weekday_nights     0.666667
adults             0.666667
children           0.666667
room_type          1.333333
daily_rate         1.333333
booking_channel    1.333333
is_canceled        0.000000
dtype: float64


In [280]:
#취소예약과 미취소 예약의 건수 및 전체예약 대비 비율 출력
booking_raw['is_canceled'].value_counts()

is_canceled
0    230
1     70
Name: count, dtype: int64

정제가 필요한 항목

1. 결측치를 가진 열들
2. 수치형 열 중 음수를 가지고 있는 열들

분석할 것 : 예약 경로별 취소비율, 객실 유형별 취소 비율, 입실일까지 남은 일자에 따른 취소 비율, 객실 요금에 따른 취소 비율

# 2

In [268]:
booking_clean = booking_raw.copy()

In [269]:
booking_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   booking_id       300 non-null    object 
 1   lead_time        298 non-null    float64
 2   weekend_nights   298 non-null    float64
 3   weekday_nights   298 non-null    float64
 4   adults           298 non-null    float64
 5   children         298 non-null    float64
 6   room_type        296 non-null    object 
 7   daily_rate       296 non-null    float64
 8   booking_channel  296 non-null    object 
 9   is_canceled      300 non-null    int64  
dtypes: float64(6), int64(1), object(3)
memory usage: 23.6+ KB


In [ ]:
# 2. 다음 조건에 해당하는 행을 찾아 예약 번호와 오류 값을 출력합니다.
#   - `lead_time`, `weekend_nights`, `weekday_nights`, `adults`, `children` 중 하나라도 음수인 경우
#   - `daily_rate`가 0 이하인 경우
for row in booking_clean.itertuples() :
    if ( row.adults < 0 ) :
        print(row.booking_id, row.Index)

booking_clean.loc[booking_clean['adults'] < 0, 'adults'] = np.nan
booking_clean.loc[booking_clean['weekend_nights'] < 0, 'weekend_nights'] = np.nan
booking_clean.loc[booking_clean['weekday_nights'] < 0, 'weekday_nights'] = np.nan
booking_clean.loc[booking_clean['adults'] < 0, 'adults'] = np.nan
booking_clean.loc[booking_clean['children'] < 0, 'children'] = np.nan

booking_clean['room_type'] = booking_clean['room_type'].fillna("Unknown")
booking_clean['booking_channel'] = booking_clean['booking_channel'].fillna("Unknown")
booking_clean['daily_rate'] = booking_clean['daily_rate'].fillna(booking_clean['daily_rate'].median())

booking_clean.isna().sum()

# booking_clean = booking_clean.dropna()
# booking_clean.isna().sum()


HB0044 243


booking_id         0
lead_time          2
weekend_nights     3
weekday_nights     3
adults             3
children           3
room_type          0
daily_rate         0
booking_channel    0
is_canceled        0
dtype: int64

In [142]:
# booking_clean[(booking_clean['lead_time']<0) | (booking_clean['weekend_nights']<0) | (booking_clean['weekday_nights']<0) | (booking_clean['adults']<0) | (booking_clean['children']<0)]
booking_clean[(booking_clean['lead_time']<0)]

,booking_id,lead_time,weekend_nights,weekday_nights,adults,children,room_type,daily_rate,booking_channel,is_canceled
219,HB0041,-1.0,2.0,3.0,1.0,1.0,Deluxe,180000.0,Online,0


In [144]:
booking_clean[(booking_clean['weekend_nights']<0)]

,booking_id,lead_time,weekend_nights,weekday_nights,adults,children,room_type,daily_rate,booking_channel,is_canceled
173,HB0042,27.0,-1.0,1.0,2.0,0.0,Deluxe,190000.0,Direct,0


In [146]:
booking_clean[(booking_clean['weekday_nights']<0)]

,booking_id,lead_time,weekend_nights,weekday_nights,adults,children,room_type,daily_rate,booking_channel,is_canceled
25,HB0043,6.0,2.0,-1.0,3.0,1.0,Deluxe,250000.0,Direct,1


In [148]:
booking_clean[(booking_clean['adults']<0)]

,booking_id,lead_time,weekend_nights,weekday_nights,adults,children,room_type,daily_rate,booking_channel,is_canceled
243,HB0044,29.0,0.0,5.0,-1.0,0.0,Standard,130000.0,Direct,0


In [150]:
booking_clean[(booking_clean['children']<0)]

,booking_id,lead_time,weekend_nights,weekday_nights,adults,children,room_type,daily_rate,booking_channel,is_canceled
216,HB0045,74.0,2.0,2.0,2.0,-1.0,Deluxe,160000.0,Online,1
